In [ ]:
import logging
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from geopy.distance import geodesic
import random

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger()

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Define paths
google_drive_path = "/content/drive/My Drive/synthetic_logs"  # Adjust the path based on your Google Drive structure
progress_file_path = os.path.join(google_drive_path, 'progress.txt')
os.makedirs(google_drive_path, exist_ok=True)

# Define initial attack scenarios
attack_types = ["Normal Traffic", "Reconnaissance", "Brute Force", "DDoS"]
attack_durations = {
    "Normal Traffic": (1, 5),
    "Reconnaissance": (10, 30),
    "Brute Force": (20, 60),
    "DDoS": (30, 120)
}
attack_frequencies = {
    "Normal Traffic": 0.7,
    "Reconnaissance": 0.1,
    "Brute Force": 0.1,
    "DDoS": 0.1
}

# Define IP ranges
public_ips = [f"192.168.{i}.{j}" for i in range(1, 5) for j in range(1, 255)]
private_ips = [f"10.0.{i}.{j}" for i in range(1, 3) for j in range(1, 255)]

# Example GeoIP data for public IPs representing global locations
geoip_data = {
    "192.168.1.1": {"country": "US", "city": "New York", "latitude": 40.7128, "longitude": -74.0060},
    "192.168.1.2": {"country": "DE", "city": "Berlin", "latitude": 52.5200, "longitude": 13.4050},
    "192.168.1.3": {"country": "JP", "city": "Tokyo", "latitude": 35.6895, "longitude": 139.6917},
    "192.168.1.4": {"country": "IN", "city": "Mumbai", "latitude": 19.0760, "longitude": 72.8777},
    "192.168.2.1": {"country": "BR", "city": "Sao Paulo", "latitude": -23.5505, "longitude": -46.6333},
    "192.168.2.2": {"country": "ZA", "city": "Johannesburg", "latitude": -26.2041, "longitude": 28.0473},
    "192.168.3.1": {"country": "CN", "city": "Shanghai", "latitude": 31.2304, "longitude": 121.4737},
    "192.168.3.2": {"country": "AU", "city": "Sydney", "latitude": -33.8688, "longitude": 151.2093},
    "192.168.4.1": {"country": "FR", "city": "Paris", "latitude": 48.8566, "longitude": 2.3522},
    "192.168.4.2": {"country": "RU", "city": "Moscow", "latitude": 55.7558, "longitude": 37.6173}
}

# Generate more public IPs and corresponding GeoIP data
for i in range(5, 256):
    for j in range(1, 255):
        ip = f"192.168.{i}.{j}"
        geoip_data[ip] = {
            "country": random.choice(["US", "DE", "JP", "IN", "BR", "ZA", "CN", "AU", "FR", "RU"]),
            "city": "City_" + str(random.randint(1, 100)),
            "latitude": np.random.uniform(-90, 90),
            "longitude": np.random.uniform(-180, 180)
        }

# Define frequent attackers (a subset of public IPs)
frequent_attackers = random.sample(public_ips, 50)  # 50 frequent attacker IPs

# Calculate distance based on geo coordinates
def calculate_distance(src_ip, dest_ip="10.0.2.16"):
    if src_ip in geoip_data:
        src_coords = (geoip_data[src_ip]["latitude"], geoip_data[src_ip]["longitude"])
        dest_coords = (geoip_data[dest_ip]["latitude"], geoip_data[dest_ip]["longitude"]) if dest_ip in geoip_data else (0, 0)
        return geodesic(src_coords, dest_coords).kilometers
    return 0

# Function to adjust frequencies for seasonality
def adjust_frequencies_for_seasonality(hour, day):
    if day in [5, 6]:  # Weekends
        return 0.5
    if 0 <= hour < 6:  # Early morning
        return 0.3
    if 6 <= hour < 12:  # Morning
        return 0.7
    if 12 <= hour < 18:  # Afternoon
        return 1.0
    if 18 <= hour < 24:  # Evening
        return 0.8

# Function to add trend and seasonality
def apply_trend_and_seasonality(attack_frequencies, date):
    month = date.month
    if month in [6, 7, 8]:
        attack_frequencies["Normal Traffic"] -= 0.1
        attack_frequencies["Reconnaissance"] += 0.05
        attack_frequencies["Brute Force"] += 0.05
    elif month in [12, 1, 2]:
        attack_frequencies["Normal Traffic"] -= 0.05
        attack_frequencies["DDoS"] += 0.05
    total_frequency = sum(attack_frequencies.values())
    for key in attack_frequencies:
        attack_frequencies[key] /= total_frequency
    return attack_frequencies

# Function to evolve attack types
def evolve_attack_types(current_time):
    # Introduce new attack types periodically
    if current_time > datetime(2024, 2, 1):
        if "Ransomware" not in attack_types:
            attack_types.append("Ransomware")
            attack_durations["Ransomware"] = (20, 80)
            attack_frequencies["Ransomware"] = 0.05

    if current_time > datetime(2024, 5, 1):
        if "Phishing" not in attack_types:
            attack_types.append("Phishing")
            attack_durations["Phishing"] = (15, 45)
            attack_frequencies["Phishing"] = 0.05

    # Increase frequencies of existing attack types
    for key in attack_frequencies:
        if key != "Normal Traffic":
            attack_frequencies[key] += 0.01

def generate_synthetic_logs(start_time, num_records, log_type):
    data = []
    current_time = start_time
    session_id = 0
    previous_event_times = {}

    for _ in range(num_records):
        evolve_attack_types(current_time)
        adjusted_attack_frequencies = apply_trend_and_seasonality(attack_frequencies.copy(), current_time)
        attack_type = np.random.choice(attack_types, p=[adjusted_attack_frequencies[at] for at in attack_types])
        duration = np.random.randint(*attack_durations[attack_type])
        session_id += 1

        # Choose a source IP (70% public, 30% private, with higher probability for frequent attackers)
        if np.random.random() < 0.3:
            src_ip = np.random.choice(frequent_attackers)
        else:
            src_ip = np.random.choice(public_ips if np.random.random() < 0.7 else private_ips)

        # Choose a destination IP
        dest_ip = "10.0.2.16" if log_type == "amun" else "5.151.132.121"

        # Adjust attack frequency based on time of day and day of week
        hour = current_time.hour
        day = current_time.weekday()
        seasonal_adjustment = adjust_frequencies_for_seasonality(hour, day)

        # Generate logs in bursts to simulate realistic traffic patterns
        burst_size = np.random.randint(1, 10) if attack_type != "Normal Traffic" else 1

        for _ in range(burst_size):
            previous_event_time = previous_event_times.get(src_ip, current_time)
            time_since_last_event = (current_time - previous_event_time).total_seconds()
            previous_event_times[src_ip] = current_time

            # Embed additional information in the existing fields
            encoded_signature = f"{attack_type}|session_{session_id}|burst_{burst_size}"
            encoded_src_port = np.random.randint(1024, 65535) + int(time_since_last_event)
            encoded_dest_port = np.random.choice([80, 443, 22, 21, 8080, 3389]) + int(duration)

            # Embedding protocol logic
            if attack_type in ["Reconnaissance", "Brute Force"]:
                protocol = "tcp"
            elif attack_type == "DDoS":
                protocol = np.random.choice(["tcp", "udp"], p=[0.7, 0.3])
            else:
                protocol = np.random.choice(["tcp", "udp"])

            log_entry = {
                "timestamp": current_time.isoformat(),
                "src": src_ip,
                "direction": "inbound",
                "protocol": protocol,
                "ids_type": "network",
                "vendor_product": "Amun" if log_type == "amun" else "Dionaea",
                "type": f"{log_type}.events" if log_type == "amun" else "dionaea.connections",
                "app": log_type,
                "dest": dest_ip,
                "dest_port": encoded_dest_port,
                "signature": encoded_signature,
                "src_port": encoded_src_port,
                "sensor": "4debb22c-54a7-11ef-ab86-080027d27b60" if log_type == "amun" else "b1db7a84-3976-11ef-92ce-080027d27b60",
                "transport": protocol,
                "severity": np.random.choice(["low", "medium", "high"], p=[0.1, 0.3, 0.6])
            }

            # Additional fields specific to Dionaea logs
            if log_type == "dionaea":
                log_entry["dionaea_action"] = np.random.choice(["accept", "reject"])

            data.append(log_entry)
            current_time += timedelta(seconds=int(np.random.randint(1, 60) * seasonal_adjustment))  # Increment timestamp for next log within the burst

    return data

# Check for progress file
if os.path.exists(progress_file_path):
    with open(progress_file_path, 'r') as f:
        last_processed_date = datetime.strptime(f.read().strip(), "%Y-%m-%d")
    logger.info(f"Resuming from {last_processed_date}")
else:
    last_processed_date = datetime.now() - timedelta(days=180)
    logger.info(f"Starting from {last_processed_date}")

end_time = datetime.now()

current_time = last_processed_date

while current_time < end_time:
    daily_amun_data = generate_synthetic_logs(current_time, 800, "amun")
    daily_dionaea_data = generate_synthetic_logs(current_time, 200, "dionaea")
    daily_data = daily_amun_data + daily_dionaea_data
    daily_df = pd.DataFrame(daily_data)

    # Save to daily folder in Google Drive
    date_str = current_time.strftime("%Y-%m-%d")
    daily_folder = f"{google_drive_path}/{date_str}"
    os.makedirs(daily_folder, exist_ok=True)
    daily_df.to_csv(f"{daily_folder}/logs.csv", index=False)

    logger.info(f"Generated logs for {date_str}")

    # Update progress file
    with open(progress_file_path, 'w') as f:
        f.write(date_str)

    # Print monthly progress
    if current_time.day == 1:
        logger.info(f"Completed generating logs for {current_time.strftime('%B %Y')}")

    current_time += timedelta(days=1)


Mounted at /content/drive
